## Silver Layer: Bronze to Silver
Batch processing with schema inference and auto-merge.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "crypto_bronze")
dbutils.widgets.text("silver_schema", "crypto_silver")

In [0]:
import pyspark.sql.functions as F

json_schema = "symbol string, price_usd double, ts double"

df_inferred = spark.table(bronze_ticks).select(
    F.from_json(F.col("value"), json_schema).alias("data")
).select("data.*")

In [0]:
df_ticks = (df_inferred
    .withColumn("event_time", F.to_timestamp(F.col("ts")))
    .withColumn("source", F.lit("streaming"))
    .drop("ts")
    .dropDuplicates(["symbol", "event_time"])
)

In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

if spark.catalog.tableExists(silver_prices):
    target = DeltaTable.forName(spark, silver_prices)
    
    (target.alias("t")
        .merge(df_ticks.alias("s"), "t.symbol = s.symbol AND t.event_time = s.event_time")
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_ticks.write.format("delta").saveAsTable(silver_prices)